# Flatten Black Friday week's events

*Data Analysis in Python · Topic 7.1*

The store's GA4 export for Black Friday week 2025 is a file of nested events: one JSON object per line. Flatten a small sample first, then the whole file, and answer two questions: how much did each medium's sessions buy, and how many items did the events carry?

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that downloads the GA4 sample and the cell that creates `sample_events`.
2. In the first cell marked ✏️, create:
   - the `sample_table` table: `sample_events` flattened, one row per event
   - the `sample_item_rows` table: the `sample_table` table with one row per element of its `items` column
   - `sample_item_count`: the number of items in the four events, from the `sample_table` table's `items` column
3. In the second cell marked ✏️, create:
   - `records`: a list with one dictionary per line of `EVENTS_FILE`
   - the `events` table: `records` flattened, one row per event
4. In the third cell marked ✏️, create `revenue_by_medium`: the purchase events' `ecommerce.purchase_revenue`, totaled by the medium of the session they happened in, as a Series.
5. In the last cell marked ✏️, create:
   - the `item_events` table: the rows of the `events` table with at least one item
   - the `item_rows` table: the `item_events` table with one row per item
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `sample_table` table has 4 rows and a `geo.country` column, the `sample_item_rows` table has 6 rows and `sample_item_count` is `5`. `records` is a list of 12,511 dictionaries, and the `events` table has 12,511 rows and 37 columns. `revenue_by_medium` has 8 mediums: `(none)` is about $2,492.83, `email` $2,436.17 and `paid_social` $254.75. The `item_events` table has 1,665 rows and the `item_rows` table 1,799.

## Parameters

In [ ]:
import json
import urllib.request

import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
EVENTS_FILE = "events_ga4_sample.ndjson"

In [ ]:
# Download the GA4 sample to this session's files. Run it, and leave it as it is.
urllib.request.urlretrieve(DATA_URL + EVENTS_FILE, EVENTS_FILE)
print(f"Downloaded {EVENTS_FILE}")

In [ ]:
# Four events in GA4's shape. Run it, and leave it as it is.
sample_events = [
    {"event_name": "session_start", "user_pseudo_id": "4471.1730",
     "geo": {"country": "Canada", "continent": "Americas"},
     "items": []},
    {"event_name": "view_item", "user_pseudo_id": "4471.1730",
     "geo": {"country": "Canada", "continent": "Americas"},
     "items": [{"item_name": "Brightmoor Organic Cotton Sneakers", "price": 79.99, "quantity": 1}]},
    {"event_name": "add_to_cart", "user_pseudo_id": "4471.1730", "user_id": "SC3003",
     "geo": {"country": "Canada", "continent": "Americas"},
     "items": [{"item_name": "Brightmoor Organic Cotton Sneakers", "price": 79.99, "quantity": 1}]},
    {"event_name": "purchase", "user_pseudo_id": "4471.1730", "user_id": "SC3003",
     "geo": {"country": "Canada", "continent": "Americas"},
     "ecommerce": {"transaction_id": "SS200", "purchase_revenue": 254.97},
     "items": [{"item_name": "Brightmoor Organic Cotton Sneakers", "price": 79.99, "quantity": 1},
               {"item_name": "Coastline Supply Recycled Training Tee", "price": 24.99, "quantity": 1},
               {"item_name": "Quietfield Essential Parka", "price": 149.99, "quantity": 1}]},
]
len(sample_events)

## A small sample

In [ ]:
sample_table = pd.json_normalize(sample_events)
sample_item_rows = sample_table.explode("items")
sample_item_count = sample_table["items"].str.len().sum()
sample_table.shape, len(sample_item_rows), sample_item_count

## The whole file

In [ ]:
with open(EVENTS_FILE, encoding="utf-8") as file:
    records = [json.loads(line) for line in file]
events = pd.json_normalize(records)
len(records), events.shape

## Revenue by medium

In [ ]:
purchases = events[events["event_name"] == "purchase"]
revenue_by_medium = purchases.groupby("session_traffic_source_last_click.manual_campaign.medium")["ecommerce.purchase_revenue"].sum()
revenue_by_medium

## The items

In [ ]:
item_events = events[events["items"].str.len() > 0]
item_rows = item_events.explode("items")
len(item_events), len(item_rows)

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    sample_table = found.get("sample_table")
    sample_rows = found.get("sample_item_rows")
    sample_count = found.get("sample_item_count")
    if not isinstance(sample_table, pd.DataFrame):
        results.append((False, "sample_table isn't a DataFrame yet. Flatten the sample: pd.json_normalize(sample_events)."))
    elif "geo" in sample_table.columns:
        results.append((False, "The sample_table table has a geo column with dictionaries in it. Flatten it with pd.json_normalize(sample_events), without max_level=0, so geo becomes geo.country and geo.continent."))
    elif len(sample_table) != 4 or "geo.country" not in sample_table.columns:
        results.append((False, f"The sample_table table should have 4 rows, one per event, and a geo.country column. It has {len(sample_table)} rows and {list(sample_table.columns)}."))
    elif not isinstance(sample_rows, pd.DataFrame):
        results.append((False, "sample_item_rows should be the sample_table table with one row per item: sample_table.explode(\"items\")."))
    elif len(sample_rows) == 5:
        results.append((False, "The sample_item_rows table has 5 rows: the session_start event is gone. explode() keeps an event with an empty list as a row with NaN, so it should have 6. Explode the sample_table table as it is."))
    elif len(sample_rows) != 6:
        results.append((False, f"The sample_item_rows table should have 6 rows: one per item, plus one for the session_start event's empty list. It has {len(sample_rows)}."))
    elif not number(sample_count):
        results.append((False, f"sample_item_count should be a number, the items in the four events. It is {sample_count!r}."))
    elif sample_count == 6:
        results.append((False, "sample_item_count is 6: that's the sample_item_rows table's rows, and one of them is the session_start event's empty list. Count the items: sample_table[\"items\"].str.len().sum()."))
    elif sample_count == 4:
        results.append((False, "sample_item_count is 4: that's the events. Count the elements of each event's items list: sample_table[\"items\"].str.len().sum()."))
    elif sample_count != 5:
        results.append((False, f"sample_item_count should be 5: 0, 1, 1 and 3 items. It is {sample_count!r}."))
    else:
        results.append((True, "The sample has 4 events and 5 items, and explode() gave 6 rows: one per item, and one for the empty list."))

    records = found.get("records")
    events = found.get("events")
    if not isinstance(records, list) or len(records) == 0:
        results.append((False, "records should be a list with one dictionary per line of the file: [json.loads(line) for line in file], inside with open(EVENTS_FILE, encoding=\"utf-8\") as file:."))
    elif len(records) != 12511 or not isinstance(records[0], dict):
        results.append((False, f"records should have 12,511 dictionaries, one per line of the file. It has {len(records):,} values, and the first is a {type(records[0]).__name__}."))
    elif not isinstance(events, pd.DataFrame):
        results.append((False, "events isn't a DataFrame yet. Flatten the records: pd.json_normalize(records)."))
    elif "device" in events.columns:
        results.append((False, "The events table has a device column with dictionaries in it: it was read without being flattened. Create it with pd.json_normalize(records)."))
    elif pd.api.types.is_float_dtype(events.get("user_pseudo_id")):
        results.append((False, "The events table's user_pseudo_id is a float: pd.read_json() guessed its type and rounded the IDs. Flatten records with pd.json_normalize(records) instead."))
    elif len(events) != 12511 or "device.category" not in events.columns:
        results.append((False, f"The events table should have 12,511 rows, one per event, and columns such as device.category. It has {len(events):,} rows."))
    else:
        results.append((True, f"The events table has 12,511 events and {events.shape[1]} columns, from {len(records):,} lines."))

    by_medium = found.get("revenue_by_medium")
    if isinstance(by_medium, pd.DataFrame) and by_medium.shape[1] == 1:
        by_medium = by_medium.iloc[:, 0]
    if not isinstance(by_medium, pd.Series):
        results.append((False, "revenue_by_medium should be a Series: the purchases' ecommerce.purchase_revenue, grouped by session_traffic_source_last_click.manual_campaign.medium and summed."))
    elif "paid_social" not in by_medium.index or not number(by_medium.get("paid_social")):
        results.append((False, f"revenue_by_medium should have one row per medium, such as paid_social and email. Its index is {list(by_medium.index)[:8]}."))
    elif abs(by_medium["paid_social"] - 1280.03) < 0.01:
        results.append((False, "revenue_by_medium's paid_social is $1,280.03: that's traffic_source.medium, where each user first came from. Group by the session's medium: session_traffic_source_last_click.manual_campaign.medium."))
    elif by_medium.sum() == 150:
        results.append((False, "revenue_by_medium counts events. Total the purchases' ecommerce.purchase_revenue with sum()."))
    elif len(by_medium) != 8 or abs(by_medium["paid_social"] - 254.75) > 0.01 or abs(by_medium.sum() - 10198.06) > 0.01:
        results.append((False, f"revenue_by_medium should have 8 mediums that add up to $10,198.06, with paid_social $254.75. It has {len(by_medium)}, adding up to ${by_medium.sum():,.2f}."))
    else:
        results.append((True, f"Purchase revenue by the session's medium: (none) ${by_medium['(none)']:,.2f}, email ${by_medium['email']:,.2f}, and paid social only ${by_medium['paid_social']:,.2f}."))

    item_events = found.get("item_events")
    item_rows = found.get("item_rows")
    if not isinstance(item_events, pd.DataFrame):
        results.append((False, "item_events should be the events table's rows with at least one item: events[events[\"items\"].str.len() > 0]."))
    elif len(item_events) != 1665:
        results.append((False, f"The item_events table should have 1,665 rows: the view_item, add_to_cart, begin_checkout and purchase events, which carry items. It has {len(item_events):,}."))
    elif not isinstance(item_rows, pd.DataFrame):
        results.append((False, "item_rows should be the item_events table with one row per item: item_events.explode(\"items\")."))
    elif len(item_rows) == 12645:
        results.append((False, "The item_rows table has 12,645 rows: the events table exploded, with a NaN row for each of the 10,846 events without items. Explode the item_events table."))
    elif len(item_rows) != 1799 or item_rows["items"].isna().any():
        results.append((False, f"The item_rows table should have 1,799 rows, one per item, and no NaN in items. It has {len(item_rows):,}."))
    else:
        results.append((True, "1,665 events carry 1,799 items, and the item_rows table has one row per item."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
sample_table = pd.json_normalize(sample_events)
sample_item_rows = sample_table.explode("items")
sample_item_count = sample_table["items"].str.len().sum()
sample_table.shape, len(sample_item_rows), sample_item_count
```

```python
with open(EVENTS_FILE, encoding="utf-8") as file:
    records = [json.loads(line) for line in file]
events = pd.json_normalize(records)
len(records), events.shape
```

```python
purchases = events[events["event_name"] == "purchase"]
revenue_by_medium = purchases.groupby("session_traffic_source_last_click.manual_campaign.medium")["ecommerce.purchase_revenue"].sum()
revenue_by_medium
```

```python
item_events = events[events["items"].str.len() > 0]
item_rows = item_events.explode("items")
len(item_events), len(item_rows)
```

</details>